# One-click reproduction of the reported experiments

This notebook runs every active reproduction script in `experiments/code/` and refreshes the
archived JSON, CSV, report and table files in `experiments/results/`. It contains no manuscript
text, no LaTeX build step and no code that produces the figures embedded in the paper.

**Run once.** Open this notebook from the repository root and use *Kernel → Restart & Run All*.
A single cell executes the analysis scripts, prints progress, and ends with a summary and the
headline numbers read back from the regenerated summaries. The default run finishes in a few
minutes. Seven scripts that re-score all 276 real columns from the aggregated inputs are skipped
by default because they take hours. Their per-case outputs are already archived, and the next
cell shows how to include them.

**Prerequisites.** Install the dependencies first.

```bash
python3 -m pip install -r requirements.txt
```

Only `numpy`, `scipy` and `matplotlib` are needed by the scripts. `jupyter` is needed to open
this notebook. No GPU and no PyTorch are required.

**Side effect.** The scripts write back into `experiments/results/` in place. That is the point
of a reproduction run. Restore the committed archives afterwards with the following command.

```bash
git checkout -- experiments/results
```

Generated table fragments, local inspection plots and the run report are ignored by Git.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import platform
import subprocess
import sys
import time

def locate_root():
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    env = os.environ.get("PFH_ROOT")
    if env:
        candidates.insert(0, Path(env).resolve())
    for cand in candidates:
        if (cand / "experiments" / "code" / "d4d_logic.py").is_file() and (cand / "experiments" / "data").is_dir():
            return cand
    raise RuntimeError(
        "Cannot find the repository root. Open this notebook from inside PFH_recover "
        "or set the PFH_ROOT environment variable to the repository directory."
    )

ROOT = locate_root()
CODE = ROOT / "experiments" / "code"
RES = ROOT / "experiments" / "results"
print("repository root :", ROOT)
print("python          :", sys.executable, platform.python_version())
for name in ("numpy", "scipy", "matplotlib"):
    missing = importlib.util.find_spec(name) is None
    print(f"  {name:<12}: {'MISSING, run pip install -r requirements.txt' if missing else 'ok'}")
assert all(importlib.util.find_spec(n) is not None for n in ("numpy", "scipy", "matplotlib")), "Install the requirements first."

## Configuration

`SKIP` lists scripts to omit from this run. It defaults to `SLOW_SCRIPTS`, the seven scripts that
re-score all 276 real columns from the aggregated inputs. Those scripts take hours and their
per-case CSV outputs are already archived. Every analysis, paired test, diagnostic, theory check
and local plot is still re-run from those archives.

Set `SKIP = set()` to reproduce the entire chain including the full re-scoring. `TIMEOUT` is the
wall-clock limit in seconds for one script.

In [ ]:
SLOW_SCRIPTS = {"validate_baselines.py", "validate_real_data.py",
                "validate_fitted_prior.py", "validate_modern_baselines.py",
                "validate_optimizer_readout.py", "analyze_readout_2x2.py",
                "validate_use_value_and_full_support.py"}

SKIP = SLOW_SCRIPTS
TIMEOUT = 3600

STAGES = [
    [
        "Stage 1. Paired statistics and case-level analyses",
        [
            "significance_tests.py",
            "analyze_pair_unification.py",
            "analyze_readout_ablation.py",
            "analyze_hybrid_fallback.py",
            "analyze_deployment_value.py",
            "analyze_population_scale_recovery.py",
            "validate_downstream_fair_comparison.py",
            "validate_joint_recovery.py"
        ]
    ],
    [
        "Stage 2. Prior, band, scope and mechanism diagnostics",
        [
            "make_data_dependency.py",
            "analyze_prior_and_joint.py",
            "analyze_a0_boundary.py",
            "analyze_fmax_sensitivity.py",
            "analyze_scope_evidence.py",
            "analyze_review_evidence.py",
            "analyze_table2_mechanism.py",
            "analyze_band_stratified.py"
        ]
    ],
    [
        "Stage 3. Theory checks, runtime and local inspection plots",
        [
            "verify_diffusion_semigroup.py",
            "verify_modern_claims.py",
            "benchmark_runtime.py",
            "make_fof_recovery_figure.py",
            "make_hist_figures.py",
            "make_unknown_support_table.py"
        ]
    ],
    [
        "Stage 4. Optional full rescoring of the per-case metric tables",
        [
            "validate_baselines.py",
            "validate_real_data.py",
            "validate_fitted_prior.py",
            "validate_modern_baselines.py",
            "validate_optimizer_readout.py",
            "analyze_readout_2x2.py",
            "validate_use_value_and_full_support.py"
        ]
    ]
]

known = {name for _, names in STAGES for name in names}
discovered = {p.name for p in CODE.glob("*.py")}
modules = {"d4d_logic.py", "d4d_metrics.py", "d4d_baselines.py",
           "d4d_modern_baselines.py", "eb_prior.py"}
extra = sorted(discovered - known - modules)
if extra:
    STAGES.append(("Extra scripts found on disk", extra))
missing = sorted((known - discovered) - SKIP)
print("scripts scheduled :", sum(len(n) for _, n in STAGES) - len(SKIP))
print("missing on disk   :", missing if missing else "none")
print("extra discovered  :", extra if extra else "none")

## Run every experiment

Each script is started as a fresh subprocess from the repository root. Failures do not stop the
run. The last lines of standard error are printed for any script that fails, and the final cell
lists everything that needs attention.

In [ ]:
def run_script(name):
    path = CODE / name
    env = dict(os.environ)
    env["PYTHONPATH"] = str(CODE) + os.pathsep + env.get("PYTHONPATH", "")
    env["MPLBACKEND"] = "Agg"
    started = time.time()
    try:
        cp = subprocess.run([sys.executable, str(path)], cwd=str(ROOT), env=env,
                            capture_output=True, text=True, timeout=TIMEOUT)
        rc, out, err = cp.returncode, cp.stdout, cp.stderr
    except subprocess.TimeoutExpired as exc:
        out = exc.stdout or ""
        err = (exc.stderr or "") + f"\nTIMEOUT after {TIMEOUT} seconds"
        rc = -9
    except Exception as exc:
        out, err, rc = "", f"{type(exc).__name__}: {exc}", -1
    elapsed = time.time() - started
    tail = [ln for ln in (err or "").strip().splitlines() if ln.strip()][-4:]
    return dict(script=name, rc=rc, seconds=round(elapsed, 1),
                stdout_lines=len((out or "").splitlines()),
                stderr_tail=tail,
                stdout_tail=[ln for ln in (out or "").strip().splitlines() if ln.strip()][-2:])

records = []
for stage, names in STAGES:
    print("=" * 78)
    print(stage)
    print("=" * 78)
    for name in names:
        if name in SKIP:
            print(f"  SKIP  {name}")
            continue
        rec = run_script(name)
        records.append(rec)
        flag = "OK  " if rec["rc"] == 0 else "FAIL"
        print(f"  {flag}  {rec['seconds']:7.1f}s  {name}")
        for ln in rec["stdout_tail"] if rec["rc"] == 0 else rec["stderr_tail"]:
            print(" " * 11 + ln[:150])
print("=" * 78)
print("finished", len(records), "scripts in", round(sum(r["seconds"] for r in records), 1), "seconds")

## Summary and headline numbers

The report is written to `reproduction_report.json` in the repository root. The values below are
read from the regenerated summaries, so they reflect this run rather than a stored copy.

In [ ]:
failed = [r for r in records if r["rc"] != 0]
scheduled = {n for _, names in STAGES for n in names}
skipped = sorted(SKIP & scheduled)
report = {"root": str(ROOT), "python": sys.executable,
          "total": len(records), "skipped": skipped,
          "failed": [r["script"] for r in failed],
          "elapsed_seconds": round(sum(r["seconds"] for r in records), 1),
          "records": records}
(ROOT / "reproduction_report.json").write_text(json.dumps(report, indent=1) + "\n", encoding="utf-8")
print("passed", len(records) - len(failed), "of", len(records), "scripts")
print("skipped", len(skipped), "full-rescoring scripts")
if failed:
    print("FAILED", [r["script"] for r in failed])
else:
    print("All reproduction scripts completed without an error.")

HEADLINES = [
    ("pair_unification/summary.json", "paired win rate against the linear reference"),
    ("deployable_regime/summary.json", "reported case counts"),
    ("readout_2x2/summary.json", "fixed-prior 2x2 readout medians"),
    ("band_calibration/summary.json", "per-frequency band coverage"),
    ("runtime_benchmark/summary.json", "wall-clock and peak memory"),
]
print("\nRegenerated summaries")
for rel, label in HEADLINES:
    p = RES / rel
    if not p.is_file():
        print(f"  {rel:<46} MISSING")
        continue
    d = json.loads(p.read_text(encoding="utf-8"))
    keys = list(d)[:6] if isinstance(d, dict) else ["<list>"]
    print(f"  {rel:<46} {label} | keys {keys}")
print("\nArchived files now under experiments/results :", sum(1 for p in RES.rglob("*") if p.is_file()))
print("Local inspection plots :", sum(1 for p in (RES / "figures").glob("*") if p.is_file()) if (RES / "figures").is_dir() else 0)
print("Table fragments        :", sum(1 for p in (RES / "tables").glob("*") if p.is_file()) if (RES / "tables").is_dir() else 0)

## After the run

Compare the regenerated numbers with the committed archives using `git diff experiments/results`.
Deterministic scripts use fixed seeds, so differences should be limited to formatting or to
platform-dependent floating point noise. Restore the committed state with the command shown at
the top of this notebook.

To re-score the per-case metric tables as well, set `SKIP = set()` in the configuration cell and
run the notebook again. That reproduces the complete chain from the aggregated inputs.

This is the only notebook in the repository. The scripts under `experiments/code/` are plain
Python files and can also be run individually from the command line.